In [1]:
import torch
import normflows as nf

ModuleNotFoundError: No module named 'normflows'

In [3]:
def log_expectation_reward(
        self,
        t: torch.Tensor,
        x: torch.Tensor,
        energy_function,
        num_mc_samples: int,
):
    repeated_x = x.unsqueeze(0).repeat_interleave(num_mc_samples, dim=0)

    samples = self.reverse_sample(repeated_x, t)

    log_rewards = energy_function(samples)

    return torch.logsumexp(log_rewards, dim=-1) - np.log(num_mc_samples)

def estimate_grad_Rt(
        self,
        t: torch.Tensor,
        x: torch.Tensor,
        energy_function,
        num_mc_samples: int = 20,
):
    if t.ndim == 0:
        t = t.unsqueeze(0).repeat(len(x))

    grad_fxn = torch.func.grad(self.log_expectation_reward, argnums=1)
    vmapped_fxn = torch.vmap(grad_fxn, in_dims=(0, 0, None, None), randomness="different")

    return vmapped_fxn(t, x, energy_function, num_mc_samples)

In [4]:
x = torch.tensor([[1.3, 1.3]], requires_grad=True)
true_score = torch.autograd.grad(energy_func_gmm2(x), x)[0]
true_score

In [27]:
x.repeat(4, 1).unsqueeze(0).repeat()

In [33]:
K = 500
bs = 4
t = 5
samples = x - torch.randn([K, 2]) * t
samples.requires_grad_(True)
log_rewards = energy_func_gmm2(samples)
lse = torch.logsumexp(log_rewards, dim=-1)
torch.autograd.grad(lse, x)